##Imports


In [31]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

##Dataset


In [65]:
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv("/content/drive/Shareddrives/CS133 Project/dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Preprocessing


In [66]:
# Strip white space in column names
data.columns = data.columns.str.strip()

In [67]:
# Find null values (Only 4 entries have null values in Flow Bytes/s)
null_counts = data.isnull().sum()
null_counts[null_counts > 0]

,0
Flow Bytes/s,4


In [68]:
# Remove entries with null values
data.dropna(inplace=True, axis='rows')

In [69]:
# Drop Flow ID and Timestamp (not helpful in detecting DDOS)
if 'Flow ID' in data.columns:
    data.drop('Flow ID', axis=1, inplace=True)
if 'Timestamp' in data.columns:
    data.drop('Timestamp', axis=1, inplace=True)

# Drop columns that only have only one value
cols_to_drop = data.columns[data.nunique() == 1]
data.drop(cols_to_drop, axis=1, inplace=True)

In [70]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 225741 entries, 0 to 225744
Data columns (total 73 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Source IP                    225741 non-null  object 
 1   Source Port                  225741 non-null  int64  
 2   Destination IP               225741 non-null  object 
 3   Destination Port             225741 non-null  int64  
 4   Protocol                     225741 non-null  int64  
 5   Flow Duration                225741 non-null  int64  
 6   Total Fwd Packets            225741 non-null  int64  
 7   Total Backward Packets       225741 non-null  int64  
 8   Total Length of Fwd Packets  225741 non-null  int64  
 9   Total Length of Bwd Packets  225741 non-null  int64  
 10  Fwd Packet Length Max        225741 non-null  int64  
 11  Fwd Packet Length Min        225741 non-null  int64  
 12  Fwd Packet Length Mean       225741 non-null  float64
 13  Fwd 

## Machine Learning

In [71]:
target = data['Label']
features = data.drop('Label', axis=1)

In [73]:
le = LabelEncoder()

# Apply LabelEncoder to IP address columns
features['Source IP'] = le.fit_transform(features['Source IP'])
features['Destination IP'] = le.fit_transform(features['Destination IP'])

# Display the encoded columns
display(features[['Source IP', 'Destination IP']].head())

# Replace infinity values with the maximum finite value for each column
features.replace([np.inf, -np.inf], np.nan, inplace=True)
features.fillna(features.max(), inplace=True)

,Source IP,Destination IP
0,22,866
1,37,866
2,37,866
3,45,860
4,49,866


# Machine Learning: DecisionTree Classifier

In [75]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_dex, test_dex in split.split(features, target):
    X_train, X_test = features.iloc[train_dex], features.iloc[test_dex]
    y_train, y_test = target.iloc[train_dex], target.iloc[test_dex]

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [74]:
from sklearn.tree import DecisionTreeClassifier

# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
result = clf.score(X_test, y_test)
print(f"Decision Tree Accuracy: {result}")

Decision Tree Accuracy: 0.9999557022303927


In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
result = rf.score(X_test, y_test)
print(f"Random Forest Accuracy: {result}")

0.9999557022303927


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(X_train, y_train)

result = lr.score(X_test, y_test)
print(f"Logistic Regression Accuracy: {result}")

Logistic Regression Accuracy: 0.9994462778799087
